# Achtung, die Kurve! - PPO Training (Colab)

Trains the CNN+PPO agent in `ai/` through the five curriculum phases (solo -> self-play -> multiplayer -> league -> items), fully headless (no rendering, pure NumPy/Pillow rasterization) so this runs at the environment's real simulation speed. Each phase is one `train.py` call; later phases warm-start from the previous phase's checkpoint with `--init-from`.

**Before running**: push your local repo (including the new `ai/` folder) to GitHub, then set `REPO_URL` below. Runtime -> Change runtime type -> GPU.

In [ ]:
!nvidia-smi

Also check how many CPU cores this runtime actually has - the environment simulation itself runs on CPU (only the PPO network updates use the GPU), and `n_envs` parallel worker processes competing for fewer cores than that will make the first rollout (before any log line prints) take a while. **If training looks silent for several minutes, that's normal on a CPU-constrained runtime - watch the progress bar rather than assuming it's frozen.** Set `N_ENVS` below to roughly match the core count.

In [ ]:
!nproc
N_ENVS = 4  # match this to the core count above; passed as --n-envs "$N_ENVS" to every phase call below

## 1. Setup

In [ ]:
REPO_URL = "https://github.com/jeremiassaur-2002/achtung-die-kurve_modified.git"
REPO_DIR = "/content/achtung-die-kurve"

import os
if not os.path.exists(REPO_DIR):
    !git clone "$REPO_URL" "$REPO_DIR"
%cd $REPO_DIR

In [ ]:
!pip install -q -r ai/requirements.txt

In [ ]:
# Mount Drive so checkpoints/logs survive Colab session resets - all runs write under ai/runs,
# so pointing --run-root at a Drive path is enough to persist everything across sessions.
from google.colab import drive
drive.mount("/content/drive")

RUN_ROOT = "/content/drive/MyDrive/achtung_kurve_runs"
import pathlib
pathlib.Path(RUN_ROOT).mkdir(parents=True, exist_ok=True)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "$RUN_ROOT"

## 2. Phase 1 - solo survival
Wall + own-trail avoidance, no opponents, no items. Increase `--timesteps` for a real run (the config's own `total_timesteps` is used if you omit the flag).

In [ ]:
!python -m ai.training.train --config ai/config/phase1.yaml --run-root "$RUN_ROOT" --run-name phase1 --n-envs "$N_ENVS"

## 3. Phase 2 - self-play (1 opponent, recent snapshots of self)

In [ ]:
PHASE1_CKPT = f"{RUN_ROOT}/phase1/final_model.zip"
!python -m ai.training.train --config ai/config/phase2.yaml --init-from "$PHASE1_CKPT" --run-root "$RUN_ROOT" --run-name phase2 --n-envs "$N_ENVS"

## 4. Phase 3 - multiplayer curriculum (2..5 opponents)

In [ ]:
PHASE2_CKPT = f"{RUN_ROOT}/phase2/final_model.zip"
!python -m ai.training.train --config ai/config/phase3.yaml --init-from "$PHASE2_CKPT" --run-root "$RUN_ROOT" --run-name phase3 --n-envs "$N_ENVS"

## 5. Phase 4 - league training (checkpoints + Elo)

In [ ]:
PHASE3_CKPT = f"{RUN_ROOT}/phase3/final_model.zip"
!python -m ai.training.train --config ai/config/phase4.yaml --init-from "$PHASE3_CKPT" --run-root "$RUN_ROOT" --run-name phase4 --n-envs "$N_ENVS"

## 6. Phase 5 - items active

In [ ]:
PHASE4_CKPT = f"{RUN_ROOT}/phase4/final_model.zip"
!python -m ai.training.train --config ai/config/phase5.yaml --init-from "$PHASE4_CKPT" --run-root "$RUN_ROOT" --run-name phase5 --n-envs "$N_ENVS"

## 7. Evaluate
Win rate / survival / placement / kills / item usage vs. random, every rule-based difficulty, and the league history.

In [ ]:
FINAL_CKPT = f"{RUN_ROOT}/phase5/final_model.zip"
LEAGUE_DIR = f"{RUN_ROOT}/phase5/league"
!python -m ai.evaluation.evaluate --checkpoint "$FINAL_CKPT" --league "$LEAGUE_DIR" --matches 30

## 8. Export weights
ONNX for `ai_bot.js` (in-browser inference via onnxruntime-web) + the native SB3 checkpoint + a plain PyTorch state_dict, then download them locally.

In [ ]:
EXPORT_DIR = f"{RUN_ROOT}/phase5/exported"
!python -m ai.export.export_onnx --checkpoint "$FINAL_CKPT" --out "$EXPORT_DIR/model.onnx"
!python -m ai.export.export_weights --checkpoint "$FINAL_CKPT" --out-dir "$EXPORT_DIR"

In [ ]:
# Download to your machine (files already persist on Drive at EXPORT_DIR regardless -
# use this if you want them locally right away instead of syncing Drive).
from google.colab import files
files.download(f"{EXPORT_DIR}/model.onnx")
files.download(f"{EXPORT_DIR}/model_sb3.zip")
files.download(f"{EXPORT_DIR}/policy_state_dict.pt")

## Next steps

- Copy `model.onnx` into the game's repo root and load it from `ai_bot.js` (see that file's top-of-file comment for the exact path/setup).
- To keep training later, re-run any phase's cell with `--init-from` pointing at the latest checkpoint on Drive - everything under `RUN_ROOT` (metrics, league, self-play pool, reports) persists across sessions.